# WE3DS — YOLOv8s Weed Detection (Local GPU)

| | |
|---|---|
| **Dataset** | WE3DS — 2568 images, 1600×1144, semantic segmentation masks + depth maps |
| **Model** | YOLOv8s |
| **Classes** | `crop` (broad bean, pea, corn, soybean, sunflower, sugar beet) · `weed` (all others) |
| **Target** | Bounding-box centre `(cx, cy)` → laser aim point |

**Pipeline:** Environment → Dataset Analysis → Mask Conversion → Training → Curves → Validation → Laser Target Demo → NCNN Export

## 1 · Environment

In [ ]:
import torch, sys, subprocess
from pathlib import Path

SCRIPT_DIR  = Path().resolve()
DATASET_DIR = SCRIPT_DIR / 'WE3DS'
YOLO_DIR    = SCRIPT_DIR / 'yolo_we3ds'
RUNS_DIR    = SCRIPT_DIR / 'runs'

print('='*55)
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU     : {gpu.name}')
    print(f'VRAM    : {gpu.total_memory/1e9:.1f} GB')
    print(f'CUDA    : {torch.version.cuda}')
else:
    print('GPU     : NOT FOUND — check CUDA installation')
print('='*55)
print(f'Dataset : {DATASET_DIR.exists()}')
print(f'WE3DS/  : {list(DATASET_DIR.iterdir()) if DATASET_DIR.exists() else "NOT FOUND"}')

## 2 · Dataset Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image

with open(DATASET_DIR / 'class_names.txt') as f:
    CLASS_NAMES = [l.strip() for l in f if l.strip()]

with open(DATASET_DIR / 'train.txt') as f:
    train_ids = [l.strip() for l in f if l.strip()]
with open(DATASET_DIR / 'test.txt') as f:
    val_ids = [l.strip() for l in f if l.strip()]

imgs   = list((DATASET_DIR / 'images').glob('*.png'))
masks  = list((DATASET_DIR / 'annotations' / 'segmentation' / 'SegmentationLabel').glob('*.png'))
depths = list((DATASET_DIR / 'depth_refined').glob('*.png'))

print(f'Classes : {len(CLASS_NAMES)}')
print(f'Images  : {len(imgs)}')
print(f'Masks   : {len(masks)}')
print(f'Depths  : {len(depths)}')
print(f'Train   : {len(train_ids)}')
print(f'Val     : {len(val_ids)}')

# Sample image size
sample = Image.open(imgs[0])
print(f'Res     : {sample.size[0]}×{sample.size[1]} px')

In [ ]:
# Class pixel distribution across 50 random masks
import random

mask_dir = DATASET_DIR / 'annotations' / 'segmentation' / 'SegmentationLabel'
sample_masks = random.sample(list(mask_dir.glob('*.png')), min(50, len(masks)))

counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
for mp in sample_masks:
    m = np.array(Image.open(mp))
    for i in range(len(CLASS_NAMES)):
        counts[i] += (m == i).sum()

counts_no_bg = counts.copy()
counts_no_bg[0] = 0   # hide void
counts_no_bg[1] = 0   # hide soil

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# All classes
axes[0].bar(CLASS_NAMES, counts, color='steelblue')
axes[0].set_title('Pixel count per class (50 sample masks)', fontsize=12)
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Pixel count')

# Plant classes only
plant_names  = CLASS_NAMES[2:]
plant_counts = counts[2:]
CROP_IDX_LOCAL = {0, 4, 9, 12, 13, 16}  # relative to plant_names (subtract 2)
colors = ['#2ecc71' if i in CROP_IDX_LOCAL else '#e74c3c' for i in range(len(plant_names))]
axes[1].bar(plant_names, plant_counts, color=colors)
axes[1].set_title('Plant classes (green=crop, red=weed)', fontsize=12)
axes[1].set_xticklabels(plant_names, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Pixel count')

plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'analysis_class_distribution.png', dpi=120)
plt.show()

In [ ]:
# Visualise 6 sample images + their masks side by side
with open(DATASET_DIR / 'class_colors.txt') as f:
    color_lines = [l.strip() for l in f if l.strip()]
PALETTE = [tuple(map(int, l.split())) for l in color_lines]

def colorise_mask(mask: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for idx, color in enumerate(PALETTE):
        rgb[mask == idx] = color
    return rgb

sample_ids = random.sample(train_ids, 6)
fig, axes = plt.subplots(6, 2, figsize=(14, 26))

for row, img_id in enumerate(sample_ids):
    img  = cv2.cvtColor(cv2.imread(str(DATASET_DIR / 'images' / f'img_{img_id}.png')), cv2.COLOR_BGR2RGB)
    mask = np.array(Image.open(DATASET_DIR / 'annotations' / 'segmentation' / 'SegmentationLabel' / f'img_{img_id}.png'))
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'img_{img_id} — RGB', fontsize=9)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(colorise_mask(mask))
    axes[row, 1].set_title(f'img_{img_id} — Semantic mask', fontsize=9)
    axes[row, 1].axis('off')

plt.suptitle('Sample images + semantic masks', fontsize=13, y=1.001)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'analysis_sample_images.png', dpi=100)
plt.show()

## 3 · Class Mapping & Mask → YOLO Conversion

In [ ]:
from tqdm.notebook import tqdm

CROP_INDICES   = {2, 6, 11, 14, 15, 18}   # broad_bean, pea, corn, soybean, sunflower, sugar_beet
IGNORE_INDICES = {0, 1}                    # void, soil
YOLO_CLASSES   = ['crop', 'weed']
MIN_AREA_PX    = 400

def semantic_to_yolo(idx: int):
    if idx in IGNORE_INDICES: return None
    return 0 if idx in CROP_INDICES else 1

print(f'{"idx":>4}  {"WE3DS class":<30}  YOLO label')
print('-' * 52)
for i, name in enumerate(CLASS_NAMES):
    yc = semantic_to_yolo(i)
    label = YOLO_CLASSES[yc] if yc is not None else 'ignore'
    marker = '🌾' if yc == 0 else ('🌿' if yc == 1 else '  ')
    print(f'{i:>4}  {name:<30}  {marker}  {label}')

In [ ]:
import yaml

def mask_to_labels(mask_path: Path) -> list:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return []
    h, w = mask.shape
    lines = []
    for cls_idx in np.unique(mask):
        yolo_cls = semantic_to_yolo(int(cls_idx))
        if yolo_cls is None: continue
        binary = (mask == cls_idx).astype(np.uint8)
        n, labels_im = cv2.connectedComponents(binary)
        for inst in range(1, n):
            pts = np.argwhere(labels_im == inst)
            if len(pts) < MIN_AREA_PX: continue
            r0, c0 = pts.min(axis=0)
            r1, c1 = pts.max(axis=0)
            cx = ((c0 + c1) / 2) / w
            cy = ((r0 + r1) / 2) / h
            bw = (c1 - c0) / w
            bh = (r1 - r0) / h
            lines.append(f'{yolo_cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    return lines


from tqdm import tqdm

stats = {}
for split, ids in [('train', train_ids), ('val', val_ids)]:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
    imgs_done = boxes_done = 0
    print(f'Converting {split} ({len(ids)} images)...')
    for img_id in tqdm(ids, ncols=80):
        img_src  = DATASET_DIR / 'images' / f'img_{img_id}.png'
        mask_src = DATASET_DIR / 'annotations' / 'segmentation' / 'SegmentationLabel' / f'img_{img_id}.png'
        if not img_src.exists() or not mask_src.exists(): continue
        img_dst = YOLO_DIR / 'images' / split / f'img_{img_id}.png'
        if not img_dst.exists():
            img_dst.symlink_to(img_src)
        label_lines = mask_to_labels(mask_src)
        lbl_dst = YOLO_DIR / 'labels' / split / f'img_{img_id}.txt'
        lbl_dst.write_text('\n'.join(label_lines))
        imgs_done  += 1
        boxes_done += len(label_lines)
    stats[split] = (imgs_done, boxes_done)
    print(f'  {imgs_done} images  |  {boxes_done} boxes  |  avg {boxes_done/max(imgs_done,1):.1f}/img\n')

cfg = {
    'path' : str(YOLO_DIR),
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : len(YOLO_CLASSES),
    'names': YOLO_CLASSES,
}
CFG_PATH = YOLO_DIR / 'dataset.yaml'
CFG_PATH.write_text(yaml.dump(cfg, sort_keys=False))
print(f'dataset.yaml written to {CFG_PATH}')

In [ ]:
# BBox statistics from converted labels
widths, heights, areas, classes = [], [], [], []

for lbl in (YOLO_DIR / 'labels' / 'train').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        c, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
        widths.append(w);  heights.append(h)
        areas.append(w * h);  classes.append(c)

classes = np.array(classes)
n_crop = (classes == 0).sum()
n_weed = (classes == 1).sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths,  bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('BBox width distribution (normalised)', fontsize=11)
axes[0].set_xlabel('Width (0–1)');  axes[0].set_ylabel('Count')

axes[1].hist(heights, bins=50, color='coral',     edgecolor='white')
axes[1].set_title('BBox height distribution (normalised)', fontsize=11)
axes[1].set_xlabel('Height (0–1)')

axes[2].bar(['crop', 'weed'], [n_crop, n_weed],
            color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
axes[2].set_title(f'Class balance  (total {n_crop+n_weed:,} boxes)', fontsize=11)
axes[2].set_ylabel('Box count')
for i, v in enumerate([n_crop, n_weed]):
    axes[2].text(i, v + 50, f'{v:,}', ha='center', fontsize=11)

plt.suptitle('Training label statistics', fontsize=13)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'analysis_label_stats.png', dpi=120)
plt.show()

print(f'Avg bbox area : {np.mean(areas):.5f}  (fraction of image)')
print(f'Crop boxes    : {n_crop:,}')
print(f'Weed boxes    : {n_weed:,}')
print(f'Imbalance     : 1 : {n_weed/max(n_crop,1):.1f}')

In [ ]:
# Verify: overlay YOLO bboxes on 6 training images
COLORS = {0: (46, 204, 113), 1: (231, 76, 60)}   # crop=green, weed=red

sample_ids = random.sample(train_ids, 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for ax, img_id in zip(axes, sample_ids):
    img_path = YOLO_DIR / 'images' / 'train' / f'img_{img_id}.png'
    lbl_path = YOLO_DIR / 'labels' / 'train' / f'img_{img_id}.txt'
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
    for line in lines:
        parts = line.split()
        if len(parts) < 5: continue
        cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
        x1 = int((cx - bw/2) * w);  y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w);  y2 = int((cy + bh/2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[cls], 2)
        cv2.drawMarker(img, (int(cx*w), int(cy*h)), COLORS[cls],
                       cv2.MARKER_CROSS, 12, 2)
    ax.imshow(img)
    n_crop_img = sum(1 for l in lines if l and l[0]=='0')
    n_weed_img = sum(1 for l in lines if l and l[0]=='1')
    ax.set_title(f'img_{img_id}  |  crop:{n_crop_img}  weed:{n_weed_img}', fontsize=9)
    ax.axis('off')

legend = [mpatches.Patch(color=np.array(COLORS[0])/255, label='crop'),
          mpatches.Patch(color=np.array(COLORS[1])/255, label='weed')]
fig.legend(handles=legend, loc='lower center', ncol=2, fontsize=12)
plt.suptitle('Converted YOLO labels  (+ = laser target centre)', fontsize=12)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'analysis_verify_labels.png', dpi=120)
plt.show()

## 4 · Training

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data         = str(CFG_PATH),
    imgsz        = 640,
    epochs       = 100,
    batch        = 8,        # RTX 3070 8 GB safe at 640px
    patience     = 20,
    device       = 0,
    project      = str(RUNS_DIR),
    name         = 'we3ds_yolov8s',
    exist_ok     = True,
    # Augmentation
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    flipud       = 0.5,
    fliplr       = 0.5,
    mosaic       = 1.0,
    degrees      = 10,
    translate    = 0.1,
    scale        = 0.5,
    save_period  = 10,
    workers      = 4,
    cache        = False,
)

BEST_WEIGHTS = RUNS_DIR / 'we3ds_yolov8s' / 'weights' / 'best.pt'
print(f'\nBest weights: {BEST_WEIGHTS}')

## 5 · Training Curves

In [ ]:
results_csv = RUNS_DIR / 'we3ds_yolov8s' / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print('Columns:', df.columns.tolist())
df.head(3)

In [ ]:
epoch = df['epoch']

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# Box loss
axes[0,0].plot(epoch, df['train/box_loss'], label='train', color='#3498db')
axes[0,0].plot(epoch, df['val/box_loss'],   label='val',   color='#e67e22')
axes[0,0].set_title('Box Loss');  axes[0,0].set_xlabel('Epoch');  axes[0,0].legend()

# Cls loss
axes[0,1].plot(epoch, df['train/cls_loss'], label='train', color='#3498db')
axes[0,1].plot(epoch, df['val/cls_loss'],   label='val',   color='#e67e22')
axes[0,1].set_title('Classification Loss');  axes[0,1].set_xlabel('Epoch');  axes[0,1].legend()

# DFL loss
axes[0,2].plot(epoch, df['train/dfl_loss'], label='train', color='#3498db')
axes[0,2].plot(epoch, df['val/dfl_loss'],   label='val',   color='#e67e22')
axes[0,2].set_title('DFL Loss');  axes[0,2].set_xlabel('Epoch');  axes[0,2].legend()

# mAP@0.5
axes[1,0].plot(epoch, df['metrics/mAP50(B)'],     color='#2ecc71', linewidth=2)
axes[1,0].set_title('mAP@0.5');  axes[1,0].set_xlabel('Epoch');  axes[1,0].set_ylim(0, 1)
axes[1,0].axhline(df['metrics/mAP50(B)'].max(), color='gray', ls='--', alpha=0.7,
                  label=f'max={df["metrics/mAP50(B)"].max():.3f}')
axes[1,0].legend()

# Precision & Recall
axes[1,1].plot(epoch, df['metrics/precision(B)'], label='Precision', color='#9b59b6')
axes[1,1].plot(epoch, df['metrics/recall(B)'],    label='Recall',    color='#1abc9c')
axes[1,1].set_title('Precision & Recall');  axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylim(0, 1);  axes[1,1].legend()

# mAP@0.5:0.95
axes[1,2].plot(epoch, df['metrics/mAP50-95(B)'], color='#e74c3c', linewidth=2)
axes[1,2].set_title('mAP@0.5:0.95');  axes[1,2].set_xlabel('Epoch');  axes[1,2].set_ylim(0, 1)
axes[1,2].axhline(df['metrics/mAP50-95(B)'].max(), color='gray', ls='--', alpha=0.7,
                  label=f'max={df["metrics/mAP50-95(B)"].max():.3f}')
axes[1,2].legend()

plt.suptitle('Training Curves — WE3DS YOLOv8s', fontsize=14)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'training_curves.png', dpi=130)
plt.show()

In [ ]:
best_idx = df['metrics/mAP50(B)'].idxmax()
best_row = df.iloc[best_idx]

print('─'*45)
print(f'Best epoch          : {int(best_row["epoch"])}')
print(f'mAP@0.5             : {best_row["metrics/mAP50(B)"]:.4f}')
print(f'mAP@0.5:0.95        : {best_row["metrics/mAP50-95(B)"]:.4f}')
print(f'Precision           : {best_row["metrics/precision(B)"]:.4f}')
print(f'Recall              : {best_row["metrics/recall(B)"]:.4f}')
print(f'Val box loss        : {best_row["val/box_loss"]:.4f}')
print('─'*45)

## 6 · Validation

In [ ]:
model_val = YOLO(str(BEST_WEIGHTS))
metrics   = model_val.val(data=str(CFG_PATH), imgsz=640, device=0)

print('─'*40)
print(f'mAP@0.5       : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95  : {metrics.box.map:.4f}')
print(f'Precision     : {metrics.box.mp:.4f}')
print(f'Recall        : {metrics.box.mr:.4f}')
print('─'*40)

In [ ]:
# Ultralytics saves confusion_matrix.png inside the run dir — display it
cm_path = RUNS_DIR / 'we3ds_yolov8s' / 'confusion_matrix_normalized.png'
if not cm_path.exists():
    cm_path = RUNS_DIR / 'we3ds_yolov8s' / 'confusion_matrix.png'

if cm_path.exists():
    img_cm = plt.imread(str(cm_path))
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(img_cm)
    ax.axis('off')
    ax.set_title('Confusion Matrix (normalised)', fontsize=13)
    plt.tight_layout()
    plt.savefig(SCRIPT_DIR / 'validation_confusion_matrix.png', dpi=120)
    plt.show()
else:
    print('Confusion matrix not found — run validation first.')

In [ ]:
# PR curve saved by ultralytics
pr_path = RUNS_DIR / 'we3ds_yolov8s' / 'PR_curve.png'
if pr_path.exists():
    img_pr = plt.imread(str(pr_path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img_pr)
    ax.axis('off')
    ax.set_title('Precision-Recall Curve', fontsize=13)
    plt.tight_layout()
    plt.savefig(SCRIPT_DIR / 'validation_pr_curve.png', dpi=120)
    plt.show()

In [ ]:
f1_path = RUNS_DIR / 'we3ds_yolov8s' / 'F1_curve.png'
if f1_path.exists():
    img_f1 = plt.imread(str(f1_path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img_f1)
    ax.axis('off')
    ax.set_title('F1-Confidence Curve', fontsize=13)
    plt.tight_layout()
    plt.savefig(SCRIPT_DIR / 'validation_f1_curve.png', dpi=120)
    plt.show()

In [ ]:
# Run inference on 8 val images and display predictions
model_inf = YOLO(str(BEST_WEIGHTS))
val_imgs  = random.sample(list((YOLO_DIR / 'images' / 'val').glob('*.png')), 8)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.ravel()

for ax, img_path in zip(axes, val_imgs):
    preds = model_inf.predict(str(img_path), imgsz=640, conf=0.35, verbose=False)
    res   = preds[0]
    img   = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w  = img.shape[:2]

    n_crop_p = n_weed_p = 0
    for box in res.boxes:
        cls = int(box.cls[0])
        conf_v = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cx_px = (x1 + x2) // 2
        cy_px = (y1 + y2) // 2
        color = COLORS[cls]
        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img, f'{YOLO_CLASSES[cls]} {conf_v:.2f}',
                    (x1, max(y1-6, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
        cv2.drawMarker(img, (cx_px, cy_px), color, cv2.MARKER_CROSS, 10, 2)
        if cls == 0: n_crop_p += 1
        else:        n_weed_p += 1

    ax.imshow(img)
    ax.set_title(f'{img_path.stem}\ncrop:{n_crop_p} weed:{n_weed_p}', fontsize=8)
    ax.axis('off')

legend = [mpatches.Patch(color=np.array(COLORS[0])/255, label='crop'),
          mpatches.Patch(color=np.array(COLORS[1])/255, label='weed')]
fig.legend(handles=legend, loc='lower center', ncol=2, fontsize=11)
plt.suptitle('Validation Predictions  (+ = laser target)', fontsize=13)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'validation_predictions.png', dpi=120)
plt.show()

In [ ]:
# Confidence threshold sweep: precision, recall, F1 vs conf
thresholds = np.arange(0.10, 0.95, 0.05)
precisions, recalls, f1s = [], [], []

sweep_imgs = list((YOLO_DIR / 'images' / 'val').glob('*.png'))[:200]  # sample 200

for conf_t in tqdm(thresholds, desc='Conf sweep'):
    tp = fp = fn = 0
    for img_path in sweep_imgs:
        lbl_path = YOLO_DIR / 'labels' / 'val' / f'{img_path.stem}.txt'
        gt_weeds = sum(1 for l in lbl_path.read_text().splitlines() if l and l[0]=='1')
        preds = model_inf.predict(str(img_path), imgsz=640, conf=float(conf_t), verbose=False)
        pred_weeds = sum(1 for b in preds[0].boxes if int(b.cls[0])==1)
        # simplified TP/FP/FN (box count, not IoU)
        matched = min(pred_weeds, gt_weeds)
        tp += matched
        fp += max(pred_weeds - gt_weeds, 0)
        fn += max(gt_weeds - pred_weeds, 0)
    p  = tp / max(tp + fp, 1)
    r  = tp / max(tp + fn, 1)
    f1 = 2*p*r / max(p+r, 1e-9)
    precisions.append(p); recalls.append(r); f1s.append(f1)

best_t = thresholds[np.argmax(f1s)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions, label='Precision', color='#9b59b6')
ax.plot(thresholds, recalls,    label='Recall',    color='#1abc9c')
ax.plot(thresholds, f1s,        label='F1',        color='#e74c3c', linewidth=2)
ax.axvline(best_t, color='gray', ls='--', alpha=0.8, label=f'Best conf={best_t:.2f}')
ax.set_xlabel('Confidence threshold');  ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Confidence threshold (weed class)', fontsize=12)
ax.set_xlim(0.1, 0.9);  ax.set_ylim(0, 1)
ax.legend();  ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'validation_conf_sweep.png', dpi=120)
plt.show()
print(f'Recommended confidence threshold: {best_t:.2f}')

## 7 · Laser Target Demo

In [ ]:
def get_laser_targets(image_path, depth_path=None, conf=0.35):
    """
    Returns list of dicts:
        class, cx_px, cy_px, cx_norm, cy_norm, depth_mm, conf
    """
    preds    = model_inf.predict(str(image_path), imgsz=640, conf=conf, verbose=False)
    result   = preds[0]
    orig_h, orig_w = result.orig_shape

    depth_map = None
    if depth_path and Path(depth_path).exists():
        depth_map = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED)  # 16-bit mm

    targets = []
    for box in result.boxes:
        cls_id  = int(box.cls[0])
        conf_v  = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx_px   = int((x1 + x2) / 2)
        cy_px   = int((y1 + y2) / 2)
        cx_norm = cx_px / orig_w
        cy_norm = cy_px / orig_h
        depth_mm = None
        if depth_map is not None:
            dh, dw = depth_map.shape[:2]
            depth_mm = int(depth_map[int(cy_norm*dh), int(cx_norm*dw)])
        targets.append(dict(cls=YOLO_CLASSES[cls_id], cx_px=cx_px, cy_px=cy_px,
                            cx_norm=round(cx_norm,4), cy_norm=round(cy_norm,4),
                            depth_mm=depth_mm, conf=round(conf_v,3)))
    return targets


demo_id    = val_ids[0]
demo_img   = DATASET_DIR / 'images'        / f'img_{demo_id}.png'
demo_depth = DATASET_DIR / 'depth_refined' / f'img_{demo_id}.png'

targets = get_laser_targets(demo_img, demo_depth)
weeds   = [t for t in targets if t['cls'] == 'weed']

print(f'Image  : img_{demo_id}.png')
print(f'Total detections : {len(targets)}')
print(f'Weed targets     : {len(weeds)}')
print()
print(f'{"#":<4} {"class":<6} {"cx_px":>6} {"cy_px":>6}  {"cx_norm":>8} {"cy_norm":>8}  {"depth_mm":>9}  conf')
print('-'*65)
for i, t in enumerate(targets):
    print(f'{i+1:<4} {t["cls"]:<6} {t["cx_px"]:>6} {t["cy_px"]:>6}  '
          f'{t["cx_norm"]:>8.4f} {t["cy_norm"]:>8.4f}  '
          f'{str(t["depth_mm"]):>9}  {t["conf"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: RGB with laser targets
img_rgb = cv2.cvtColor(cv2.imread(str(demo_img)), cv2.COLOR_BGR2RGB)
h, w    = img_rgb.shape[:2]
for t in targets:
    color = (46, 204, 113) if t['cls'] == 'crop' else (231, 76, 60)
    cv2.drawMarker(img_rgb, (t['cx_px'], t['cy_px']), color, cv2.MARKER_CROSS, 20, 3)
    label = f"{t['cls']} {t['depth_mm']}mm" if t['depth_mm'] else t['cls']
    cv2.putText(img_rgb, label, (t['cx_px']+8, t['cy_px']-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
axes[0].imshow(img_rgb)
axes[0].set_title(f'Laser targets — img_{demo_id}\n× = aim point', fontsize=11)
axes[0].axis('off')

# Right: depth map with same targets
if demo_depth.exists():
    depth = cv2.imread(str(demo_depth), cv2.IMREAD_UNCHANGED).astype(np.float32)
    depth_vis = (depth / depth.max() * 255).astype(np.uint8)
    depth_color = cv2.applyColorMap(depth_vis, cv2.COLORMAP_INFERNO)
    depth_color = cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB)
    dh, dw = depth.shape
    for t in weeds:
        dx = int(t['cx_norm'] * dw)
        dy = int(t['cy_norm'] * dh)
        cv2.drawMarker(depth_color, (dx, dy), (255, 255, 0), cv2.MARKER_CROSS, 20, 3)
        cv2.putText(depth_color, f"{t['depth_mm']}mm", (dx+8, dy-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,0), 2)
    axes[1].imshow(depth_color)
    axes[1].set_title('Refined depth map — weed targets\n× = 3-D laser aim point', fontsize=11)
    axes[1].axis('off')
    cb = plt.colorbar(plt.cm.ScalarMappable(cmap='inferno'), ax=axes[1], fraction=0.03)
    cb.set_label('Depth (closer → darker)')
else:
    axes[1].set_visible(False)

plt.suptitle('Laser Targeting Visualisation', fontsize=14)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'laser_target_demo.png', dpi=120)
plt.show()

## 8 · NCNN Export (Raspberry Pi)

In [ ]:
model_export = YOLO(str(BEST_WEIGHTS))
model_export.export(format='ncnn', imgsz=640)

ncnn_dir = BEST_WEIGHTS.parent / 'best_ncnn_model'
print(f'NCNN model: {ncnn_dir}')
print('Files:', [f.name for f in ncnn_dir.iterdir()])
print('\nCopy the best_ncnn_model/ folder to the Raspberry Pi.')

## 9 · Summary

In [ ]:
best_idx = df['metrics/mAP50(B)'].idxmax()
best_row = df.iloc[best_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
table_data = [
    ['Dataset',          'WE3DS'],
    ['Total images',     f"{len(train_ids)+len(val_ids):,}"],
    ['Train / Val',      f"{len(train_ids):,} / {len(val_ids):,}"],
    ['Classes',          'crop · weed'],
    ['Model',            'YOLOv8s'],
    ['Best epoch',       str(int(best_row['epoch']))],
    ['mAP@0.5',          f"{best_row['metrics/mAP50(B)']:.4f}"],
    ['mAP@0.5:0.95',     f"{best_row['metrics/mAP50-95(B)']:.4f}"],
    ['Precision',        f"{best_row['metrics/precision(B)']:.4f}"],
    ['Recall',           f"{best_row['metrics/recall(B)']:.4f}"],
    ['Weights',          str(BEST_WEIGHTS.relative_to(SCRIPT_DIR))],
    ['NCNN model',       'runs/we3ds_yolov8s/weights/best_ncnn_model/'],
]
tbl = ax.table(cellText=table_data, colLabels=['Metric', 'Value'],
               cellLoc='left', loc='center', colWidths=[0.35, 0.65])
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 1.5)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50');  cell.set_text_props(color='white')
    elif r % 2 == 0:
        cell.set_facecolor('#ecf0f1')
ax.set_title('Training Summary', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(SCRIPT_DIR / 'training_summary.png', dpi=130)
plt.show()